# Dynamic EWMA Pool Analysis

This notebook compares the **DynamicEWMA** dual-rate EWMA pool strategy
against baseline strategies and the simpler **DynamicSystem** (CV-based)
strategy on two PyPy serverless functions: **BFS** and **MST**.

## Key Difference from DynamicSystem
- **DynamicSystem** uses raw coefficient of variation (CV) from a sliding window.
  A function with inherently high variance always gets a large pool.
- **DynamicEWMA** uses a dual-rate EWMA of absolute deviation. It compares
  *recent* volatility (fast EWMA) against *baseline* volatility (slow EWMA).
  A function with consistently high but stable variance has ratio ≈ 1.0 and
  gets a moderate pool. Only *spikes above baseline* trigger pool growth.

## Modular Strategies
All strategies are defined in `STRATEGIES`. To add a new one, add one dict entry.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import math
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
from scipy.stats import gmean

In [ ]:
# ── Seaborn styling ─────────────────────────────────────────────────
sns.set()
sns.set_context("poster", font_scale=1.25)
sns.set_style("ticks")

## 1 — Configuration

In [ ]:
# ── Experiment configuration ────────────────────────────────────────

BENCHMARKS = ['bfs', 'mst']

INCLUDE_JAVA = False
JAVA_BENCHMARKS = ['matrix-multiplication', 'word-count', 'simple-hash', 'html-rendering']

# ── Strategy registry (modular) ─────────────────────────────────────
STRATEGIES = {
    'Cold Start':       'cold',
    'Fixed':            'fixed&request_to_checkpoint=1',
    'Request Centric':  'request_centric&max_capacity=12',
    'Dynamic System':   'dynamic_system',
    'Dynamic EWMA':     'dynamic_ewma',
}

BASELINE_STRATEGY_LABEL = 'Fixed'
EVAL_STRATEGY_LABEL = 'Dynamic EWMA'

EVICTION_RATES = [1, 4, 20]
MUTABILITIES = [1]

DF_COLUMNS = [
    'request_number', 'benchmark', 'mutability',
    'strategy', 'rate', 'client', 'server', 'overhead',
]

BENCHMARK_TITLES = {
    'bfs': 'BFS',
    'mst': 'MST',
    'matrix-multiplication': 'MatrixMult',
    'word-count': 'WordCount',
    'simple-hash': 'Hash',
    'html-rendering': 'HTML Rendering',
}

STRATEGY_LABEL_MAP = {v: k for k, v in STRATEGIES.items()}

print(f"Benchmarks:  {BENCHMARKS}")
print(f"Strategies:  {list(STRATEGIES.keys())}")
print(f"Baseline:    {BASELINE_STRATEGY_LABEL}")
print(f"Evaluating:  {EVAL_STRATEGY_LABEL}")
print(f"Include JVM: {INCLUDE_JAVA}")

## 2 — Data Loading

In [ ]:
def load_csv_safe(path: str) -> pd.DataFrame:
    try:
        return pd.read_csv(path, names=DF_COLUMNS)
    except FileNotFoundError:
        print(f"⚠ File not found: {path}  (skipping)")
        return pd.DataFrame(columns=DF_COLUMNS)

dfs_to_concat = [
    load_csv_safe('../data/python-evaluation.csv'),
    load_csv_safe('../data/python-evaluation-dynamic-system.csv'),
    load_csv_safe('../data/python-evaluation-dynamic-ewma.csv'),
]

if INCLUDE_JAVA:
    dfs_to_concat.append(load_csv_safe('../data/java-evaluation.csv'))
    dfs_to_concat.append(load_csv_safe('../data/java-evaluation-dynamic-system.csv'))
    dfs_to_concat.append(load_csv_safe('../data/java-evaluation-dynamic-ewma.csv'))
    BENCHMARKS = BENCHMARKS + JAVA_BENCHMARKS

df_all = pd.concat(dfs_to_concat, ignore_index=True)
df_all = df_all[df_all['benchmark'].isin(BENCHMARKS)]

known_strategies = set(STRATEGIES.values())
df_all = df_all[df_all['strategy'].isin(known_strategies)]
df_all['strategy_label'] = df_all['strategy'].map(STRATEGY_LABEL_MAP)

print(f"\nLoaded {len(df_all)} rows")
print(f"Benchmarks present: {sorted(df_all['benchmark'].unique())}")
print(f"Strategies present: {sorted(df_all['strategy_label'].dropna().unique())}")
print(f"Rates present:      {sorted(df_all['rate'].unique())}")

## 3 — Convergence Analysis

In [ ]:
def find_convergence_point(
    df, benchmark, strategy_csv, eviction_rate,
    total_requests=500, tail_fraction=0.8, tolerance=0.02,
    window_size=20, skip_initial=100,
):
    subset = df[
        (df['benchmark'] == benchmark)
        & (df['strategy'] == strategy_csv)
        & (df['rate'] == eviction_rate)
    ]
    if subset.empty:
        return None
    latencies = subset['client'].to_numpy()
    tail_start = int(tail_fraction * total_requests)
    target = np.median(latencies[latencies.shape[0] - (total_requests - tail_start):])
    lo, hi = target * (1 - tolerance), target * (1 + tolerance)
    for idx in range(skip_initial, len(latencies)):
        if lo <= np.median(latencies[idx: idx + window_size]) <= hi:
            return idx
    return None

convergence_rows = []
for benchmark in BENCHMARKS:
    for label, strat_csv in STRATEGIES.items():
        for rate in EVICTION_RATES:
            conv = find_convergence_point(df_all, benchmark, strat_csv, rate)
            convergence_rows.append({
                'benchmark': benchmark, 'strategy': label,
                'rate': rate, 'convergence_request': conv,
            })

df_convergence = pd.DataFrame(convergence_rows)
print(df_convergence.to_string(index=False))

## 4 — Strategy Comparison (rate=1)

In [ ]:
df_rate1 = df_all[df_all['rate'] == 1].copy()

df_grouped = (
    df_rate1.groupby(['benchmark', 'strategy_label'])
    .median(numeric_only=True)['client'].reset_index()
)
df_pivot = df_grouped.pivot(index='benchmark', columns='strategy_label', values='client')

baseline_col = BASELINE_STRATEGY_LABEL
eval_col = EVAL_STRATEGY_LABEL

for label in STRATEGIES:
    if label != baseline_col and label in df_pivot.columns:
        df_pivot[f'{label} imp%'] = (
            (df_pivot[baseline_col] - df_pivot[label])
            / df_pivot[baseline_col] * 100
        )

print("Median client latency (rate=1) and improvement vs", baseline_col)
print(df_pivot.round(2).to_string())

In [ ]:
THRESHOLD_POS = 5.0
THRESHOLD_NEG = -5.0

imp_col = f'{eval_col} imp%'

if imp_col in df_pivot.columns:
    improved  = df_pivot[df_pivot[imp_col] > THRESHOLD_POS].index.tolist()
    on_par    = df_pivot[(df_pivot[imp_col] >= THRESHOLD_NEG) & (df_pivot[imp_col] <= THRESHOLD_POS)].index.tolist()
    worsened  = df_pivot[df_pivot[imp_col] < THRESHOLD_NEG].index.tolist()

    print(f"Improved  (>{THRESHOLD_POS}%):  {improved}")
    print(f"On par:                {on_par}")
    print(f"Worsened  (<{THRESHOLD_NEG}%): {worsened}")

    positive = df_pivot[df_pivot[imp_col] > THRESHOLD_POS][imp_col]
    if len(positive) > 0:
        print(f"\nGeometric mean of positive improvements: {gmean(positive):.2f}%")
else:
    print(f"Column '{imp_col}' not found — ensure {eval_col} data is loaded.")

## 5 — Request-Rate Analysis

In [ ]:
df_grouped_all = (
    df_all.groupby(['benchmark', 'strategy_label', 'rate'])
    .median(numeric_only=True)['client'].reset_index()
)
df_pivot_all = df_grouped_all.pivot(
    index=['benchmark', 'rate'], columns='strategy_label', values='client'
)

if baseline_col in df_pivot_all.columns and eval_col in df_pivot_all.columns:
    df_pivot_all['improvement'] = (
        (df_pivot_all[baseline_col] - df_pivot_all[eval_col])
        / df_pivot_all[baseline_col] * 100
    )
    print(f"Improvement of {eval_col} vs {baseline_col}:")
    print(df_pivot_all['improvement'].round(2).to_string())
else:
    missing = [c for c in [baseline_col, eval_col] if c not in df_pivot_all.columns]
    print(f"Missing: {missing}")

In [ ]:
if 'improvement' in df_pivot_all.columns:
    df_pos = df_pivot_all[df_pivot_all['improvement'] > THRESHOLD_POS].reset_index()
    if not df_pos.empty:
        print(f"Geometric mean per rate:\n{df_pos.groupby('rate')['improvement'].apply(gmean)}")
    else:
        print("No benchmarks exceeded the positive threshold.")

## 6 — Head-to-head: Dynamic System vs Dynamic EWMA

Direct comparison of the two dynamic strategies.

In [ ]:
ds_col = 'Dynamic System'
de_col = 'Dynamic EWMA'

if ds_col in df_pivot.columns and de_col in df_pivot.columns:
    comparison = df_pivot[[ds_col, de_col]].copy()
    comparison['EWMA vs System %'] = (
        (comparison[ds_col] - comparison[de_col])
        / comparison[ds_col] * 100
    )
    print("Head-to-head (rate=1): positive = EWMA is faster")
    print(comparison.round(2).to_string())
else:
    missing = [c for c in [ds_col, de_col] if c not in df_pivot.columns]
    print(f"Missing columns for head-to-head: {missing}")

## 7 — CDF Visualisation

In [ ]:
fig, axes = plt.subplots(1, len(BENCHMARKS), figsize=(10 * len(BENCHMARKS), 8), squeeze=False)

for col_idx, benchmark in enumerate(BENCHMARKS):
    ax = axes[0, col_idx]
    bm_df = df_rate1[df_rate1['benchmark'] == benchmark]

    for label in STRATEGIES:
        strat_df = bm_df[bm_df['strategy_label'] == label]
        if strat_df.empty:
            continue
        sorted_vals = np.sort(strat_df['client'].values)
        cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
        ax.plot(sorted_vals, cdf, label=label)

    title = BENCHMARK_TITLES.get(benchmark, benchmark)
    ax.set_title(title)
    ax.set_xlabel('Client-side latency (µs)')
    ax.set_ylabel('CDF')
    ax.legend(fontsize='small')

plt.tight_layout()
plt.savefig('cdf_dynamic_ewma.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: cdf_dynamic_ewma.png")